In [20]:
import pandas as pd
import pyarrow.parquet as pq
import time
from pathlib import Path
import dask.dataframe as dd
import xarray as xr

In [2]:
st = time.time()
data_dir="~/W_Projects/Quant_Trading/live_data"
date='2025-08-20'
data_dir = Path(data_dir).expanduser()
print(data_dir)
parquet_files = sorted(data_dir.glob(f"*{date}_*.parquet"))
print(len(files))
et = time.time()
print(f'total time ={st-et}')

/home/willse/W_Projects/Quant_Trading/live_data
288
total time =-0.003345966339111328


In [10]:
st = time.time()
dfs = []
for f in files:
    df = pd.read_parquet(f)
    timestamp = f.stem.split('_')[-1]
    df['timestamp'] = pd.to_datetime(f'{date} {timestamp}')
    dfs.append(df)

big_df = pd.concat(dfs,ignore_index=False)
big_df = big_df.reset_index()
clean_df = big_df[big_df['ident'] != 'errors']

xr_data = clean_df.set_index(['timestamp','ident']).to_xarray()

et = time.time()
print(f'total time ={st-et}')

total time =-7.350394248962402


In [15]:
st = time.time()
ddf = dd.read_parquet(f'{data_dir}/*.parquet',engine='pyarrow')
et = time.time()
print(f'total time ={st-et}')

total time =-0.10394859313964844


In [11]:
xr_data

<xarray.Dataset> Size: 912MB
Dimensions:                             (timestamp: 288, ident: 5142)
Coordinates:
  * timestamp                           (timestamp) datetime64[ns] 2kB 2025-0...
  * ident                               (ident) object 41kB 'A' 'AA' ... 'ZYXI'
Data variables: (12/77)
    index                               (timestamp, ident) int64 12MB 680 ......
    assetMainType                       (timestamp, ident) object 12MB 'EQUIT...
    assetSubType                        (timestamp, ident) object 12MB 'COE' ...
    quoteType                           (timestamp, ident) object 12MB 'NBBO'...
    realtime                            (timestamp, ident) object 12MB True ....
    ssid                                (timestamp, ident) float64 12MB 4.844...
    ...                                  ...
    regular.regularMarketTradeTime      (timestamp, ident) float64 12MB 1.756...
    reference.fsiCode                   (timestamp, ident) object 12MB None ....
    reference.fsiDesc                   (timestamp, ident) object 12MB None ....
    invalidSymbols                      (timestamp, ident) object 12MB None ....
    reference.htbQuantity               (timestamp, ident) float64 12MB nan ....
    reference.otcMarketTier             (timestamp, ident) object 12MB None ....

In [25]:
ddf.compute()

KeyError: "['reference.htbQuantity'] not in index"

In [18]:
xr_data.to_zarr('jtest_03_out_01.zarr')

/home/willse/W_Projects/Quant_Trading/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:228: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [24]:
xr_data_2 = xr.open_dataset('jtest_03_out_01.zarr', engine='zarr')

In [23]:
xr_data_2

<xarray.Dataset> Size: 902MB
Dimensions:                             (timestamp: 288, ident: 5142)
Coordinates:
  * ident                               (ident) object 41kB 'A' 'AA' ... 'ZYXI'
  * timestamp                           (timestamp) datetime64[ns] 2kB 2025-0...
Data variables: (12/77)
    fundamental.divYield                (timestamp, ident) float64 12MB ...
    assetSubType                        (timestamp, ident) object 12MB ...
    fundamental.divAmount               (timestamp, ident) float64 12MB ...
    reference.htbRate                   (timestamp, ident) float64 12MB ...
    extended.quoteTime                  (timestamp, ident) float64 12MB ...
    fundamental.fundLeverageFactor      (timestamp, ident) float64 12MB ...
    ...                                  ...
    fundamental.eps                     (timestamp, ident) float64 12MB ...
    regular.regularMarketTradeTime      (timestamp, ident) float64 12MB ...
    reference.fsiCode                   (timestamp, ident) object 12MB ...
    reference.fsiDesc                   (timestamp, ident) object 12MB ...
    reference.otcMarketTier             (timestamp, ident) object 12MB ...
    quote.totalVolume                   (timestamp, ident) float64 12MB ...